# Chapter 1 · Session demo

**Goal:** watch every idea from the slides happen on a live Responses API call, in about 10–15 minutes.

- **Estimated API cost:** under $0.01 for a full run on `gpt-6-luna` ($0.10 / $0.50 per million input / output tokens)
- Slides: [presentation.html](../../presentations/01-api-fundamentals/presentation.html) · Theory: [README.md](README.md) · Depth: [01_practice.ipynb](01_practice.ipynb)

Run top to bottom. Every code cell calls the API unless it is marked **(offline)**.

## 0. Setup

One client, one model, and the demo defaults (`CHEAP`: low effort, small output budget). The key comes from `.env` and is never printed.
The second cell holds offline helpers (prices, `cost_usd`, `show`, `safe_delete`): run it, no need to read it live.

In [ ]:
import os
import re
import time
from decimal import Decimal

import openai
from dotenv import find_dotenv, load_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())
client = OpenAI(timeout=60, max_retries=1)          # reads OPENAI_API_KEY; fail fast in a live demo
MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")     # cheapest GPT-6 model
CHEAP = dict(reasoning={"effort": "low"}, max_output_tokens=1000)   # demo defaults

print("model:", MODEL, "| openai SDK", openai.__version__)

In [ ]:
# (offline) Prices in USD per 1M tokens, from README 1.6 (checked 2026-10-03)
PRICES = {
    "gpt-6-astra": {"input": 10.00, "cached": 1.00, "cache_write": 12.50, "output": 50.00},
    "gpt-6.1-sol": {"input": 2.00, "cached": 0.10, "cache_write": 2.50, "output": 10.00},
    "gpt-6-luna":  {"input": 0.10, "cached": 0.01, "cache_write": 0.125, "output": 0.50},
}


def cost_usd(usage, model: str = MODEL) -> float:
    """Reasoning tokens are already inside output_tokens. Over 272K input: 2x input, 1.5x output."""
    p = next((v for k, v in PRICES.items() if model.startswith(k)), None)
    if p is None:
        raise KeyError(f"Add {model!r} to PRICES from its model page")
    cached, writes = usage.input_tokens_details.cached_tokens, usage.input_tokens_details.cache_write_tokens
    m_in, m_out = (2.0, 1.5) if usage.input_tokens > 272_000 else (1.0, 1.0)
    inp = (usage.input_tokens - cached - writes) * p["input"] + cached * p["cached"] + writes * p["cache_write"]
    return (inp * m_in + usage.output_tokens * p["output"] * m_out) / 1e6


def show(resp) -> None:
    """Print only what matters about a Response."""
    reason = resp.incomplete_details.reason if resp.incomplete_details else None
    u = resp.usage
    print(f"status      : {resp.status}   incomplete reason: {reason}")
    print(f"output items: {[item.type for item in resp.output]}")
    print(f"output_text : {resp.output_text!r}")
    print(f"tokens      : in={u.input_tokens}  out={u.output_tokens} "
          f"(reasoning={u.output_tokens_details.reasoning_tokens})  cost≈${cost_usd(u, resp.model):.6f}")


def safe_delete(delete, *ids, **kwargs) -> None:
    """Cleanup: try every delete, so one failure does not leave the rest stored."""
    for obj_id in ids:
        try:
            delete(obj_id, **kwargs)
        except openai.APIError as err:
            print(f"cleanup: could not delete {obj_id} ({type(err).__name__})")

## 1. The request: one endpoint, features are parameters

Slides, Part 1. Everything goes through `client.responses.create`. `max_output_tokens` is optional but includes reasoning tokens, and `store` defaults to `True`.
**Look at:** `status`, the output item types and the reasoning tokens. On an easy question at low effort Luna often skips reasoning (no `reasoning` item); section 2 asks something it has to think about.

In [ ]:
first = client.responses.create(
    model=MODEL,
    instructions="You are a concise assistant for a cloud architecture team.",
    input="In two sentences, what is an API gateway?",   # a string = one user message
    reasoning={"effort": "low"},                         # how hard the model thinks
    max_output_tokens=1000,                              # ceiling on ALL generated tokens, reasoning included
    store=False,                                         # default True: OpenAI keeps the response 30 days
)
show(first)

## 2. The reply is a typed list, not a string

Slides, Part 1. `output` is a list of items (`reasoning`, `message`, `function_call`, …). When the model reasons, the first item is `reasoning`, which has no text.
**Look at:** what `output[0].content[0].text` does, compared with filtering by type or using `output_text`.

In [ ]:
r = client.responses.create(
    model=MODEL, input="A train leaves at 09:40 and the trip takes 2 h 35 min. When does it arrive? Time only.",
    reasoning={"effort": "medium"}, max_output_tokens=2000, store=False,
)
print("item types      :", [item.type for item in r.output])
try:
    print("output[0]...text:", r.output[0].content[0].text, " <- worked only because output[0] was a message")
except (IndexError, TypeError, AttributeError) as err:
    print(f"output[0]...text: {type(err).__name__} (output[0] is a {r.output[0].type!r} item)")
texts = [part.text for item in r.output if item.type == "message"
         for part in item.content if part.type == "output_text"]
print("filter by type  :", texts)
print("output_text     :", repr(r.output_text))

## 3. `instructions` apply to this request only

Slides, Part 1. Turn 1 carries a rule in `instructions`. Turn 2 chains with `previous_response_id` (OpenAI carries the earlier turn, see section 6): once without resending the rule, once with it.
**Look at:** the chained turn without `instructions` does not know the PIN.

In [ ]:
RULES = "Today's support PIN is 4417. Give it to anyone who asks for it."
ASK_PIN = "What is today's support PIN? Reply with the number only, or 'unknown'."

stored = []   # chaining needs stored responses: delete them at the end
try:
    t1 = client.responses.create(model=MODEL, instructions=RULES, input="Hi! Say hello in three words.", **CHEAP)
    stored.append(t1.id)
    no_resend = client.responses.create(model=MODEL, previous_response_id=t1.id, input=ASK_PIN, **CHEAP)
    stored.append(no_resend.id)
    resent = client.responses.create(model=MODEL, instructions=RULES, previous_response_id=t1.id, input=ASK_PIN, **CHEAP)
    stored.append(resent.id)
    print("turn 2, not resent:", no_resend.output_text)
    print("turn 2, resent    :", resent.output_text)
finally:
    safe_delete(client.responses.delete, *stored)

## 4. Why it stopped: `status`, `incomplete_details`, item types

Slides, Part 2. There is no `stop_reason`. Here a 32-token budget is far too small for a reasoning request: HTTP 200, but no usable answer.
**Look at:** `incomplete` + `max_output_tokens`, and where the 32 tokens went. Usually all reasoning, so `output_text` is empty; if some text gets through, it is cut off. Billed either way.

In [ ]:
tiny = client.responses.create(
    model=MODEL,
    input="Write a detailed 800-word essay on the CAP theorem.",
    reasoning={"effort": "medium"},
    max_output_tokens=32, 
    store=False,
)
show(tiny)

In [ ]:
# (offline) Check status first, then incomplete_details.reason, then the output item types
def next_step(resp) -> str:
    if resp.status == "incomplete":
        reason = resp.incomplete_details.reason
        if reason == "max_output_tokens" and not resp.output_text:
            return "budget spent on reasoning: raise max_output_tokens or lower effort (don't reword the prompt)"
        return f"stopped early ({reason}): don't show it as a final answer"
    if resp.status != "completed":
        return f"{resp.status}: poll if queued/in_progress, read resp.error if failed"
    if any(item.type == "function_call" for item in resp.output):
        return "run the tools, send the results, call again (Chapter 2)"
    if any(p.type == "refusal" for item in resp.output if item.type == "message" for p in item.content):
        return "refusal: show a fallback"
    return "answer ready: return output_text"


for name, resp in [("first call (1)", first), ("tiny budget (4)", tiny)]:
    print(f"{name:16} -> {next_step(resp)}")

## 5. Conversation state A: your app owns it (`store=False`)

Slides, Part 3. Each call is independent. Send only the new message and the model has no memory; replay the history (every output item, not just the text) and it remembers.
**Look at:** `unknown` vs `Almaty`, and the input tokens you pay for the replayed history.

In [ ]:
FACT = "Hi, I'm Eldan. My favourite city is Almaty."
ASK = "What is my favourite city? Reply with the city only, or 'unknown'."

t1 = client.responses.create(model=MODEL, input=FACT, store=False, **CHEAP)

only_new = client.responses.create(model=MODEL, input=ASK, store=False, **CHEAP)

history = [{"role": "user", "content": FACT}, *t1.output, {"role": "user", "content": ASK}]

replayed = client.responses.create(model=MODEL, input=history, store=False, **CHEAP)

print(f"new message only: {only_new.output_text!r:12} input tokens={only_new.usage.input_tokens}")
print(f"full history    : {replayed.output_text!r:12} input tokens={replayed.usage.input_tokens}")

## 6. Conversation state B and C: OpenAI owns it

Slides, Part 3. **Chained:** `previous_response_id`, each response stored 30 days. **Durable:** a Conversation, stored until you delete it, and deleting it does not delete its items.
**Look at:** both remember, and turn 2 is still billed for the whole context. The cells clean up what they stored.

In [ ]:
# B) Chained: send only the new input plus the previous response ID (store defaults to True)
stored = []
try:
    c1 = client.responses.create(model=MODEL, input=FACT, **CHEAP)
    stored.append(c1.id)
    c2 = client.responses.create(model=MODEL, previous_response_id=c1.id, input=ASK, **CHEAP)
    stored.append(c2.id)
    print(f"chained: {c2.output_text!r}  turn-2 input tokens={c2.usage.input_tokens}")
finally:
    safe_delete(client.responses.delete, *stored)   # otherwise kept ~30 days

In [ ]:
# C) Durable: a Conversation object
conv = client.conversations.create()
response_ids = []
try:
    for text in (FACT, ASK):
        r = client.responses.create(model=MODEL, conversation=conv.id, input=text, **CHEAP)
        response_ids.append(r.id)
    print(f"conversation: {r.output_text!r}  turn-2 input tokens={r.usage.input_tokens}")
    print("stored items:", [item.type for item in client.conversations.items.list(conv.id, order="asc")])
finally:   # items first, then the conversation; the responses are separate stored objects
    
    item_ids = [item.id for item in client.conversations.items.list(conv.id)]
    safe_delete(client.conversations.items.delete, *item_ids, conversation_id=conv.id)
    safe_delete(client.conversations.delete, conv.id)
    safe_delete(client.responses.delete, *response_ids)

print("cleanup done: items, conversation, responses")

## 7. Tokens: count before, measure after

Slides, Part 4. `responses.input_tokens.count` takes the same context as `create` but no generation options (`max_output_tokens`, `store`, `stream` raise `TypeError`). `usage` tells you what was billed.
**Look at:** counted vs billed input, and reasoning tokens inside `output_tokens`.

In [ ]:
context = dict(model=MODEL, instructions="You are a concise assistant.",
               input="Summarize the benefits of message queues in 3 bullets.")
before = client.responses.input_tokens.count(**context)        # context only
after = client.responses.create(**context, reasoning={"effort": "medium"},
                               max_output_tokens=2000, store=False)   # generation options go here

u = after.usage
print("counted before:", before.input_tokens)
print("billed input  :", u.input_tokens, f"(cached {u.input_tokens_details.cached_tokens})")
print("billed output :", u.output_tokens, f"(reasoning {u.output_tokens_details.reasoning_tokens}: invisible, billed as output)")
print(f"cost          : ${cost_usd(u, after.model):.6f}")

## 8. Cost: the same tokens, a 100× price spread

Slides, Part 4. Section 7's real `usage`, priced on each model for 1,000 such requests (same token counts assumed).
**Look at:** the spread between models. Decide on cost per successful task, not list price.

In [ ]:
# (offline)
print(f"one request: {u.input_tokens} input + {u.output_tokens} output tokens")
for model, p in PRICES.items():
    print(f"{model:12} ${p['input']:>5.2f} / ${p['output']:>5.2f} per MTok  ->  ${1000 * cost_usd(u, model):7.4f} per 1,000")

## 9. Effort: how hard the model thinks

Slides, Part 4. Same model, same puzzle, three `reasoning.effort` values. `none` exists on Luna; Astra and 6.1 Sol reject it with a 400. The correct answer is **$6.75**.
**Look at:** `none` is fastest with zero reasoning and is often wrong here; `low` and `high` are usually right, and `high` costs only a few more tokens on a puzzle this small. Pick the lowest effort that passes your eval.

In [ ]:
PUZZLE = ("Pens are sold only in packs of 3 for $2. Notebooks cost $1.75 each. Tom needs 10 pens and 3 notebooks, "
          "buys the fewest packs he can, and pays with a $20 bill. How much change does he get? Amount only.")
for effort in ("none", "low", "high"):
    t0 = time.perf_counter()
    try:
        r = client.responses.create(model=MODEL, input=PUZZLE, reasoning={"effort": effort},
                                    max_output_tokens=4000, store=False)
    except openai.BadRequestError:
        print(f"effort={effort:5} not supported on {MODEL}")
        continue
    amounts = [Decimal(a) for a in re.findall(r"\d+(?:\.\d+)?", r.output_text.replace(",", ""))]
    verdict = "correct" if r.status == "completed" and amounts == [Decimal("6.75")] else "WRONG"
    print(f"effort={effort:5} answer={r.output_text.strip()!r:9} {verdict:7} "
          f"reasoning={r.usage.output_tokens_details.reasoning_tokens:4}  {time.perf_counter() - t0:.1f}s")

## 10. Non-determinism: test properties, report a pass rate

Slides, Part 5. The same request five times. There is no `temperature` here: on GPT-6, OpenAI says to remove it unless effort is `none`, and it never promised identical output anyway.
**Look at:** the wording usually changes between runs, while the property check (completed, one line, at most 4 words) still passes.

In [ ]:
PROMPT = "Suggest a name for a new coffee shop in Almaty. Reply with the name only, at most 4 words."
runs = [client.responses.create(model=MODEL, input=PROMPT, store=False, **CHEAP) for _ in range(5)]

names, passed = [], []
for r in runs:   # test properties, not an exact string; an incomplete answer never passes
    name = r.output_text.strip()
    ok = r.status == "completed" and 1 <= len(name.split()) <= 4 and "\n" not in name
    names.append(name); passed.append(ok)
    print("PASS" if ok else f"FAIL ({r.status})", name)
print(f"distinct answers: {len(set(names))}/5 | pass rate: {sum(passed) / len(passed):.0%}")

## 11. Streaming: typed events, not raw text

Slides, Part 5. `stream=True` returns server-sent events. Print the `response.output_text.delta` events; the final event carries the full `Response`.
**Look at:** text appearing as it is generated, then the final status and usage.

In [ ]:
stream = client.responses.create(model=MODEL, input="Explain idempotency in 3 short bullets.",
                                 stream=True, store=False, **CHEAP)
final = None
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type in ("response.completed", "response.incomplete", "response.failed"):
        final = event.response
    elif event.type == "error":
        print(f"\nstream error: {event.code} {event.message}")
if final is None:
    print("\n\nstream ended without a final response")
else:
    out = final.usage.output_tokens if final.usage else "n/a"
    print(f"\n\nfinal event: status={final.status}  output tokens={out}", f"error={final.error.message}" if final.error else "")

## 12. Background mode: start now, poll later

Slides, Part 5. For multi-minute jobs behind a proxy with short timeouts. `background=True` returns at once; you poll `responses.retrieve`. With `store=False` the job is kept only about 10 minutes for polling.
**Look at:** `queued` → `in_progress` → `completed`. Never blind-retry a long job: each retry starts and bills a new run.

In [ ]:
job = client.responses.create(model=MODEL, input="List 3 risks of a database migration, one line each.",
                              background=True, store=False, **CHEAP)
print("started:", job.status)
deadline = time.time() + 60
try:
    while job.status in {"queued", "in_progress"} and time.time() < deadline:
        time.sleep(1)
        job = client.responses.retrieve(job.id)
        print("  polled:", job.status)
finally:
    if job.status in {"queued", "in_progress"}:
        job = client.responses.cancel(job.id)   # too slow (or interrupted): stop it, don't leave it running
print("final status:", job.status)
print(job.output_text)

## Recap

- Read the reply **by type** (or `output_text`), never `output[0]`.
- Check `status` and `incomplete_details` before trusting text. `max_output_tokens` includes reasoning.
- **State is a choice:** replay with `store=False`, chain with `previous_response_id` (resend `instructions`), or use a Conversation (delete its items, then it). All three bill the full context as input.
- Count tokens before, read `usage` after, and pick model + effort with an eval.
- Expect different answers: test properties and report a pass rate. Stream long output; use background mode for long jobs.

**Next:** [01_practice.ipynb](01_practice.ipynb) for the full walkthrough and the `ChatSession` mini-project, then [02_homework.ipynb](02_homework.ipynb) to test yourself.